# V2 Embargo Robustness Control (Train-Domain, One-Day Embargo)

**Designation (verbatim, in every artifact and summary):** train-domain
one-day-embargo robustness control (train-inner domain evidence only; no model
selected).

**Research question.** The route substitutes within-day locality for purge and
embargo; the eval rows of each train-inner fold begin on the trading day right
after the train rows end, and cross-day serial dependence across that boundary
stays intact. Was that adjacency inflating the train-inner readout? Within one
run, the same fitted tcn_tiny models are read under the exact Stage 02 fold
rows (`no_embargo`) and under a one-trading-day eval-side embargo
(`embargo_1day`: drop each fold's first eval trading day; train rows
untouched).

**Shared fits (exact contrast).** The embargo changes eval rows only, so both
variants share the identical fitted model per (fold, seed): one fit, two
readouts, zero fit-level nondeterminism in the contrast. The same-row
stratified dummy is re-drawn per variant, per fold, per seed.

**INVARIANT (bold, enforced in code):** the entire experiment runs on the
frozen TRAIN segment only (1998-01-02 through 2013-09-16, end-exclusive).
ZERO contact with the official validation split. ZERO contact with post-2017
rows. The stage code raises on any timestamp at or after 2013-09-16.

**Evidence status.** Every number this notebook produces is train-inner-domain
evidence about fold-boundary adjacency. No outcome removes the paper's stated
limitation; a one-day embargo probes lag-one adjacency only.

Preregistration (frozen before any fold runs):
`docs/protocols/v2_embargo_robustness_preregistration_20260701.md`

## Preregistration Summary

- Machinery: the executed Stage 02 train-inner path unchanged — frozen Stage
  00 labels (3.0 bps, 9 bars; nothing relabeled), candidate
  `price_volume_time_w20` (w=20), 3 chronological day-block folds, Stage 02
  fold-row caps, frozen `tcn_tiny` profile `tcn_p01`, frozen seeds [101, 202].
  The no-embargo variant's capped fold rows are parity-gated against the
  executed Stage 02 plan ledger, so the baseline IS the executed Stage 02 row
  contract.
- EXACT embargo rule (frozen): drop every capped eval row whose trading day
  equals the fold's first eval trading day (the calendar day of the fold's
  eval_start). Train rows and fold boundaries untouched; the embargoed rows
  are a strict, documented subset of the baseline rows
  (`emb_dropped_rows.csv` lists every dropped row).
- Readout: same-row macro-F1 delta versus the per-variant re-drawn stratified
  dummy, per variant x fold x seed; margins are fold means per seed.
- Predeclared reading (descriptive only): the shrinkage rule applies per seed
  only when that seed's no-embargo margin is strictly positive; "materially
  smaller" = the embargoed margin falls below 0.5 x the no-embargo margin
  (more than half the baseline margin disappears); both seeds must agree.
  Materially smaller -> cross-day dependence was inflating the readout
  (limitation strengthened). Roughly unchanged -> limitation bounded at the
  one-day scale, NOT removed. Inapplicable/mixed -> reported as such, no
  verdict. Incomplete rows void the reading.
- Budget: 3 folds x 2 seeds = 6 shared fits -> 12 readout rows (2 variants x
  2 seeds = 4 readout units over the 3 folds).

FORBIDDEN wording in anything built on this run: solved problem, leakage-free
proof, purge complete, final model, clean test, profitable.

## Expected Artifacts

```
/content/lst_models_results/v2_embargo_robustness/<run_id>/
  run_manifest.json
  artifact_inventory.csv
  emb_trial_ledger.csv
  emb_variant_summary.csv
  emb_fold_manifest.csv
  emb_dropped_rows.csv
  emb_baseline_control_summary.csv
  emb_reading_readout.json
```

Durable backup: `My Drive/lst_models/results/v2_embargo_robustness/<run_id>/`

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
# Two-step exact-commit pin: push the full v2_embargo_robustness bundle
# (notebook + config + preregistration + src + tests) first, then fill
# PROJECT_REPO_COMMIT with that full-bundle commit.
PROJECT_REPO_COMMIT = "3a55664c6ef1c89a9cbe2fdcc78ce1ae411f82c2"
PROJECT_ROOT = Path("/content/lst_models")

RUN_STAGE00_DRIVE_SYNC = True
RUN_STAGE01_DRIVE_SYNC = True
RUN_STAGE02_DRIVE_SYNC = True
RUN_RAW_DATA_SYNC = True
RUN_EMB = False
RUN_EMB_DRIVE_BACKUP = True
RUN_ARTIFACT_DISPLAY = False

STAGE_NAME = "v2_embargo_robustness"
ROUTE = "lst_models"
SCOPE = "validation_only"
HOLDOUT_TEST_CONTACT = False
TRAIN_DOMAIN_ONLY = True
TRAIN_END_EXCLUSIVE = "2013-09-16"
FROZEN_SEEDS = [101, 202]
EMBARGO_VARIANT_IDS = ["no_embargo", "embargo_1day"]
EMBARGO_TRADING_DAYS = 1
SHRINKAGE_FRACTION = 0.5
STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE02_RUN_ID = "20260610_082130_797479"
EMB_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE02_OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner") / STAGE02_RUN_ID
OUTPUT_DIR = Path("/content/lst_models_results/v2_embargo_robustness")
EMB_OUTPUT_RUN_DIR = OUTPUT_DIR / EMB_RUN_ID
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE02_DRIVE_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner", STAGE02_RUN_ID]
EMB_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "v2_embargo_robustness"]
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/v2_embargo_robustness")

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "v2_embargo_robustness.yaml").exists()
        and (path / "docs" / "protocols" / "v2_embargo_robustness_preregistration_20260701.md").exists()
        and (path / "notebooks" / "v2_embargo_robustness_colab.ipynb").exists()
        and (path / "src" / "lst_models" / "stages" / "embargo_robustness.py").exists()
        and (path / "src" / "lst_models" / "robustness.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    for candidate in [Path("/content/lst_models"), Path("/content") / zip_path.stem]:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError("No lst_models project root found after zip extraction.")

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("manual_upload mode only works inside Colab.") from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "REPLACE_WITH" in PROJECT_REPO_COMMIT:
            raise ValueError(
                "Fill PROJECT_REPO_COMMIT with the v2_embargo_robustness "
                "full-bundle commit before running (two-step exact-commit pin)."
            )
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "v2_embargo_robustness.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "v2_embargo_robustness_preregistration_20260701.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "v2_embargo_robustness_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "embargo_robustness.py"
DOMAIN_MODULE_PATH = PROJECT_ROOT / "src" / "lst_models" / "robustness.py"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"
TCN_SEARCH_SPACE_PATH = PROJECT_ROOT / "configs" / "models" / "tcn" / "search_space.yaml"
REQUIRED_PROJECT_FILES = [
    STAGE_CONFIG_PATH, PROTOCOL_PATH, NOTEBOOK_PATH, STAGE_ENTRYPOINT_PATH,
    DOMAIN_MODULE_PATH, RAW_DATA_MANIFEST_PATH, TCN_SEARCH_SPACE_PATH,
]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError("v2_embargo_robustness bootstrap target is missing required files:\n" + missing_text)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("EMB_RUN_ID:", EMB_RUN_ID)

## Config Load And Contract Check

In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyYAML is required to read the v2 embargo robustness config.") from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required v2 embargo robustness file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(PROTOCOL_PATH)
require_path(RAW_DATA_MANIFEST_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    emb_config = yaml.safe_load(handle)

stage_inputs = emb_config["inputs"]
stage_outputs = emb_config["outputs"]
stage_checkpointing = emb_config["checkpointing"]
stage_inputs["stage00_run_id"] = STAGE00_RUN_ID
stage_inputs["stage00_runtime_run_dir"] = str(STAGE00_OUTPUT_DIR)
stage_inputs["stage00_drive_path_parts"] = STAGE00_DRIVE_PATH_PARTS
stage_inputs["stage01_run_id"] = STAGE01_RUN_ID
stage_inputs["stage01_runtime_run_dir"] = str(STAGE01_OUTPUT_DIR)
stage_inputs["stage01_drive_path_parts"] = STAGE01_DRIVE_PATH_PARTS
stage_inputs["stage02_run_id"] = STAGE02_RUN_ID
stage_inputs["stage02_runtime_run_dir"] = str(STAGE02_OUTPUT_DIR)
stage_inputs["stage02_drive_path_parts"] = STAGE02_DRIVE_PATH_PARTS
stage_inputs["raw_data_dir"] = str(RAW_DATA_DIR)
stage_outputs["output_dir"] = str(OUTPUT_DIR)
stage_outputs["run_id"] = EMB_RUN_ID
stage_checkpointing["checkpoint_dir"] = str(CHECKPOINT_ROOT)

assert emb_config["stage_name"] == STAGE_NAME
assert emb_config["route"] == ROUTE
assert emb_config["scope"] == SCOPE
assert emb_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert emb_config["train_domain_only"] is TRAIN_DOMAIN_ONLY
assert stage_inputs["stage00_run_id"] == STAGE00_RUN_ID
assert Path(stage_inputs["stage00_runtime_run_dir"]) == STAGE00_OUTPUT_DIR
assert stage_inputs["stage01_run_id"] == STAGE01_RUN_ID
assert Path(stage_inputs["stage01_runtime_run_dir"]) == STAGE01_OUTPUT_DIR
assert stage_inputs["stage02_run_id"] == STAGE02_RUN_ID
assert Path(stage_inputs["stage02_runtime_run_dir"]) == STAGE02_OUTPUT_DIR
assert Path(stage_inputs["raw_data_dir"]) == RAW_DATA_DIR
assert Path(stage_outputs["output_dir"]) == OUTPUT_DIR
assert stage_outputs["run_id"] == EMB_RUN_ID
assert Path(stage_checkpointing["checkpoint_dir"]) == CHECKPOINT_ROOT
assert stage_inputs["raw_data_manifest"] == "configs/lst_models_data.yaml"
assert emb_config["candidate"]["candidate_id"] == "price_volume_time_w20"
assert emb_config["model"]["family"] == "tcn"
assert emb_config["model"]["probe_id"] == "tcn_tiny"
assert emb_config["model"]["hpo_profile_id"] == "tcn_p01"
assert emb_config["embargo"]["rule_id"] == "drop_first_eval_trading_day_per_fold"
assert emb_config["embargo"]["embargo_trading_days"] == EMBARGO_TRADING_DAYS
assert emb_config["embargo"]["fits_shared_across_variants"] is True
assert [v["variant_id"] for v in emb_config["embargo"]["variants"]] == EMBARGO_VARIANT_IDS
assert emb_config["train_inner"]["n_folds"] == 3
assert emb_config["train_inner"]["seeds"] == FROZEN_SEEDS
assert emb_config["reading_rules"]["primary_baseline"] == "stratified_dummy_train_prior"
assert emb_config["reading_rules"]["shrinkage_fraction"] == SHRINKAGE_FRACTION
assert emb_config["reading_rules"]["evaluated_per_seed_and_must_agree"] is True
torch_defaults = emb_config["probe_training_defaults"]["torch"]
assert torch_defaults["early_stopping"] == "inner_train_chronological_tail"
assert torch_defaults["epochs"] == 64
assert torch_defaults["gradient_clip_norm"] > 0
planned_fit_rows = int(emb_config["train_inner"]["n_folds"]) * len(emb_config["train_inner"]["seeds"])
planned_readout_rows = planned_fit_rows * len(EMBARGO_VARIANT_IDS)
assert planned_fit_rows <= emb_config["budget"]["max_planned_fit_rows"]
assert planned_readout_rows <= emb_config["budget"]["max_readout_rows"]

from lst_models.stages.embargo_robustness import _validate_config
_validate_config(emb_config)

print(json.dumps({
    "stage_name": emb_config["stage_name"],
    "evidence_status": emb_config["evidence_status"],
    "variants": EMBARGO_VARIANT_IDS,
    "embargo_rule_id": emb_config["embargo"]["rule_id"],
    "planned_fit_rows": planned_fit_rows,
    "planned_readout_rows": planned_readout_rows,
    "candidate_id": emb_config["candidate"]["candidate_id"],
    "profile": emb_config["model"]["hpo_profile_id"],
    "seeds": emb_config["train_inner"]["seeds"],
    "train_domain_only": emb_config["train_domain_only"],
    "holdout_test_contact": emb_config["holdout_test_contact"],
}, indent=2))

## Stage 00, Stage 01, Stage 02, And Raw Data Input Sync

Exact frozen run folders only (no latest-run scanning); raw `.txt` files are
downloaded by Drive file ID from `configs/lst_models_data.yaml`. The Stage 02
folder supplies only `02_hpo_plan_ledger.csv`, the fold-row parity source for
the no-embargo variant.

In [ ]:
from lst_models.artifacts import require_artifacts

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(q=" and ".join(query_parts), spaces="drive", fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def get_drive_service_for_input_sync():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive sync only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def sync_stage_artifacts_from_drive(service, output_dir, path_parts, required_names, label):
    run_folder_id = resolve_drive_folder(service, path_parts)
    for artifact_name in required_names:
        output_path = Path(output_dir) / artifact_name
        if output_path.exists():
            continue
        file_meta = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, file_meta["id"], output_path)
        print(f"downloaded {label}: {artifact_name}")

def sync_raw_data_from_drive(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    raw_source = raw_manifest["raw_source"]
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in raw_manifest["tickers"]:
        spec = raw_source["files"][ticker]
        output_path = RAW_DATA_DIR / spec["name"]
        if output_path.exists():
            continue
        download_drive_file(service, spec["file_id"], output_path)
        print(f"downloaded raw data: {output_path.name}")

required_stage00_artifacts = emb_config["inputs"]["required_stage00_artifacts"]
required_stage01_artifacts = emb_config["inputs"]["required_stage01_artifacts"]
required_stage02_artifacts = emb_config["inputs"]["required_stage02_artifacts"]
if RUN_EMB:
    service = get_drive_service_for_input_sync()
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    missing_raw = []
    for ticker in raw_manifest["tickers"]:
        raw_path = RAW_DATA_DIR / raw_manifest["raw_source"]["files"][ticker]["name"]
        if not raw_path.exists():
            missing_raw.append(raw_path.name)
    if missing_raw and RUN_RAW_DATA_SYNC:
        print("Missing raw data files; syncing from Drive file IDs:", missing_raw)
        sync_raw_data_from_drive(service)
    stage00_missing = [name for name in required_stage00_artifacts if not (STAGE00_OUTPUT_DIR / name).exists()]
    if stage00_missing and RUN_STAGE00_DRIVE_SYNC:
        print("Missing Stage 00 artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS, stage00_missing, "stage00")
    stage01_missing = [name for name in required_stage01_artifacts if not (STAGE01_OUTPUT_DIR / name).exists()]
    if stage01_missing and RUN_STAGE01_DRIVE_SYNC:
        print("Missing Stage 01 artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS, stage01_missing, "stage01")
    stage02_missing = [name for name in required_stage02_artifacts if not (STAGE02_OUTPUT_DIR / name).exists()]
    if stage02_missing and RUN_STAGE02_DRIVE_SYNC:
        print("Missing Stage 02 plan-ledger artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE02_OUTPUT_DIR, STAGE02_DRIVE_PATH_PARTS, stage02_missing, "stage02")
    require_artifacts(STAGE00_OUTPUT_DIR, required_stage00_artifacts)
    require_artifacts(STAGE01_OUTPUT_DIR, required_stage01_artifacts)
    require_artifacts(STAGE02_OUTPUT_DIR, required_stage02_artifacts)
    print("Stage 00, Stage 01, Stage 02, and raw data input checks passed.")
else:
    print("RUN_EMB=False; input sync skipped.")

## Run The Embargo Robustness Control

Requires a GPU runtime (Runtime -> Change runtime type -> T4 GPU or better).
Expected wall clock on a T4: the raw rebuild plus one window-dataset build
dominate; with only 6 capped tcn_tiny fits (each read out under both
variants) the whole run is well inside a single session (order of an hour
end-to-end). One compact progress line prints per completed fold via the
checkpoint writer.

In [ ]:
if RUN_EMB:
    try:
        from lst_models.stages.embargo_robustness import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Missing entry point: src/lst_models/stages/embargo_robustness.py.") from exc
    result = run_stage(emb_config)
    display(result)
    if Path(result.output_dir).name != EMB_RUN_ID:
        raise RuntimeError(f"run id mismatch: {Path(result.output_dir).name} != {EMB_RUN_ID}")
    if Path(result.output_dir) != EMB_OUTPUT_RUN_DIR:
        raise RuntimeError(f"output dir mismatch: {Path(result.output_dir)} != {EMB_OUTPUT_RUN_DIR}")
else:
    print("RUN_EMB=False; embargo robustness control not executed.")

## Durable Drive Result Backup

Runs immediately after a successful `run_stage`. Validates the required
artifact list, uploads to the canonical Drive run folder, and writes
`drive_backup_manifest.json` last (self-reference recorded with null bytes).

In [ ]:
def get_drive_service_for_result_backup():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive backup only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_or_create_drive_folder(service, parent_id, name):
    escaped_name = quote_drive_query_value(name)
    folder_mime = "application/vnd.google-apps.folder"
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and mimeType = '{folder_mime}' and trashed = false"
    response = service.files().list(q=query, spaces="drive", fields="files(id, name)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) > 1:
        raise RuntimeError(f"duplicate Drive folders named {name!r} under parent {parent_id}; resolve manually")
    if matches:
        return matches[0]["id"]
    created = service.files().create(
        body={"name": name, "mimeType": folder_mime, "parents": [parent_id]}, fields="id"
    ).execute()
    return created["id"]

def upload_or_update_drive_file(service, folder_id, local_path):
    from googleapiclient.http import MediaFileUpload
    escaped_name = quote_drive_query_value(local_path.name)
    query = f"name = '{escaped_name}' and '{folder_id}' in parents and trashed = false"
    response = service.files().list(q=query, spaces="drive", fields="files(id, name)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) > 1:
        raise RuntimeError(f"duplicate Drive files named {local_path.name!r} under folder {folder_id}; resolve manually")
    media = MediaFileUpload(str(local_path), resumable=True)
    if matches:
        updated = service.files().update(fileId=matches[0]["id"], media_body=media, fields="id").execute()
        return updated["id"]
    created = service.files().create(
        body={"name": local_path.name, "parents": [folder_id]}, media_body=media, fields="id"
    ).execute()
    return created["id"]

if RUN_EMB_DRIVE_BACKUP and RUN_EMB:
    from lst_models.stages.embargo_robustness import REQUIRED_EMB_ARTIFACTS
    local_run_dir = EMB_OUTPUT_RUN_DIR
    pass
    required_backup_files = list(REQUIRED_EMB_ARTIFACTS)
    missing_required_artifacts = [name for name in required_backup_files if not (local_run_dir / name).exists()]
    if missing_required_artifacts:
        raise FileNotFoundError(
            "Missing required v2_embargo_robustness artifacts before Drive backup: "
            f"{missing_required_artifacts} in {local_run_dir}"
        )
    service = get_drive_service_for_result_backup()
    parent_id = "root"
    for folder_name in EMB_DRIVE_RESULT_PATH_PARTS + [EMB_RUN_ID]:
        parent_id = find_or_create_drive_folder(service, parent_id, folder_name)
    drive_folder_id = parent_id
    uploaded_files = []
    for name in required_backup_files:
        local_path = local_run_dir / name
        drive_file_id = upload_or_update_drive_file(service, drive_folder_id, local_path)
        uploaded_files.append({
            "file_name": name,
            "relative_path": name,
            "drive_file_id": drive_file_id,
            "bytes": local_path.stat().st_size,
        })
        print("uploaded:", name)
    backup_manifest_path = local_run_dir / "drive_backup_manifest.json"
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": EMB_RUN_ID,
        "local_output_dir": str(local_run_dir),
        "drive_path_parts": EMB_DRIVE_RESULT_PATH_PARTS + [EMB_RUN_ID],
        "drive_folder_id": drive_folder_id,
        "uploaded_files": uploaded_files + [{
            "file_name": "drive_backup_manifest.json",
            "relative_path": "drive_backup_manifest.json",
            "drive_file_id": None,
            "bytes": None,
            "self_reference": True,
        }],
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "train_domain_only": True,
        "holdout_test_contact": False,
    }
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    manifest_file_id = upload_or_update_drive_file(service, drive_folder_id, backup_manifest_path)
    backup_manifest["uploaded_files"][-1]["drive_file_id"] = manifest_file_id
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    print("stage_run_id:", EMB_RUN_ID)
    print("drive_path:", "/".join(["My Drive"] + EMB_DRIVE_RESULT_PATH_PARTS + [EMB_RUN_ID]))
    print("drive_folder_id:", drive_folder_id)
else:
    print("RUN_EMB_DRIVE_BACKUP disabled or RUN_EMB=False; Drive backup skipped.")

## Artifact Display

In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd
    variant_summary = pd.read_csv(EMB_OUTPUT_RUN_DIR / "emb_variant_summary.csv")
    display(variant_summary)
    fold_manifest = pd.read_csv(EMB_OUTPUT_RUN_DIR / "emb_fold_manifest.csv")
    display(fold_manifest[[
        "fold_id", "first_eval_trading_day", "n_capped_eval_rows",
        "n_embargo_retained_rows", "n_embargo_dropped_rows",
    ]])
    with (EMB_OUTPUT_RUN_DIR / "emb_reading_readout.json").open("r", encoding="utf-8") as handle:
        reading = json.load(handle)
    print(json.dumps({key: reading[key] for key in [
        "overall_outcome", "shrinkage_fraction", "embargo_rule_id",
        "limitation_removed", "evidence_status",
    ]}, indent=2))
    print(json.dumps(reading["per_seed"], indent=2))
else:
    print("RUN_ARTIFACT_DISPLAY=False; artifact display skipped.")

## Interpretation Guard

- Everything above is **train-inner-domain evidence about fold-boundary
  adjacency**. It is not market evidence, selects no model, and is never
  fused with the official validation, train-inner control, or guarded
  walk-forward domains.
- No outcome removes the paper's limitation: a one-day embargo probes lag-one
  cross-day adjacency only; longer-lag dependence and the actual boundary
  between train and the official readout segment remain untested.
- The only admissible reading is the preregistered shrinkage rule in
  `emb_reading_readout.json` (applies per seed only when the no-embargo
  margin is positive; both seeds must agree). A materially-smaller outcome
  strengthens the limitation; a roughly-unchanged outcome bounds it.
- FORBIDDEN wording in any summary built on this run: solved problem,
  leakage-free proof, purge complete, final model, clean test, profitable.
- Outcomes map to paper edits ONLY through the preregistration's section 9
  outcome map and the claims-ledger process; numbers may be quoted only from
  `emb_variant_summary.csv` / `emb_reading_readout.json` of the completed run.
- Any deviation (config edit, rule change, rerun) must be recorded in the
  preregistration's section 10 deviation log BEFORE results are interpreted.